In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Additional utilities
import nltk
from nltk.corpus import stopwords
import string
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Download NLTK data if needed
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')


In [ ]:
# LSTM Implementation for Language Modeling and Stock Prediction

# Task 1: LSTM for Stock Price Prediction
def create_lstm_stock_model(input_shape, units=50):
    """
    Create LSTM model for stock price prediction
    
    Args:
        input_shape: Shape of input data (time_steps, features)
        units: Number of LSTM units
        
    Returns:
        Compiled LSTM model
    """
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(units, return_sequences=True),
        Dropout(0.2),
        LSTM(units),
        Dropout(0.2),
        Dense(1, activation='linear')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Generate synthetic stock data for demonstration
def generate_stock_data(n_samples=2000, sequence_length=60):
    """Generate synthetic stock price data"""
    np.random.seed(42)
    
    # Generate stock-like data with trend and volatility
    prices = [100]  # Starting price
    for i in range(n_samples):
        # Random walk with slight upward trend
        change = np.random.normal(0.05, 1.0)  # Small positive trend with volatility
        new_price = prices[-1] * (1 + change/100)
        prices.append(max(new_price, 1))  # Ensure price stays positive
    
    prices = np.array(prices[1:])  # Remove initial price
    
    # Normalize data
    scaler = StandardScaler()
    prices_scaled = scaler.fit_transform(prices.reshape(-1, 1))
    
    # Create sequences
    X, y = [], []
    for i in range(sequence_length, len(prices_scaled)):
        X.append(prices_scaled[i-sequence_length:i, 0])
        y.append(prices_scaled[i, 0])
    
    return np.array(X), np.array(y), scaler, prices

# Generate and prepare data
print("Generating synthetic stock data...")
X_stock, y_stock, stock_scaler, original_prices = generate_stock_data()

# Reshape for LSTM input
X_stock = X_stock.reshape((X_stock.shape[0], X_stock.shape[1], 1))

# Train/test split
split_idx = int(0.8 * len(X_stock))
X_train_stock = X_stock[:split_idx]
y_train_stock = y_stock[:split_idx]
X_test_stock = X_stock[split_idx:]
y_test_stock = y_stock[split_idx:]

print(f"Stock data prepared:")
print(f"Training samples: {X_train_stock.shape[0]}")
print(f"Test samples: {X_test_stock.shape[0]}")
print(f"Sequence length: {X_train_stock.shape[1]}")

# Visualize the generated stock data
plt.figure(figsize=(12, 6))
plt.plot(original_prices[:500])
plt.title('Generated Stock Price Data (First 500 days)')
plt.xlabel('Days')
plt.ylabel('Price')
plt.grid(True)
plt.show()
